# Ridge, Lasso, and Ensembles

In *Trees and Regularization*, pruning charged a tree for its size: training error plus strength times complexity. That shape works for any model with a size to charge for. For linear regression, the size is how big the coefficients are, and charging for it gives **ridge** and **lasso** regression (ISL §6.2). As with $\alpha$, the strength is chosen by cross-validation. The second half of the notebook returns to trees, with a third way to cut variance: grow many trees and average them.

### By the end, you'll be able to

9. Write the ridge and lasso objectives, read each in words, and put them side by side with pruning's: same shape, different measure of complexity.
10. Predict how the coefficients change as $\lambda$ grows: ridge shrinks them all toward zero; lasso sets some **exactly** to zero, so it also selects features.
11. Explain why features must be standardized before ridge or lasso, and why the intercept isn't penalized.
12. Choose $\lambda$ by cross-validation and refit, and describe $\lambda$ as a flexibility setting in bias–variance terms.
13. Translate between ISL's symbols and scikit-learn's names ($\alpha$, $\lambda$, `ccp_alpha`, `alpha`, `C`, `learning_rate`).
14. Describe bagging, and explain why averaging many trees reduces variance.
15. Describe how a random forest adds one more source of randomness, and how boosting differs: small trees fit one after another, each correcting the last.
16. Say which ensemble can overfit by adding more trees (boosting) and which can't (bagging, random forests).
17. Name where this week shows up next: weight decay in a neural network is ridge.

(The numbering continues from the *Trees and Regularization* notebook.)

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as pandas (`pd.`) and scikit-learn, do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_8_demos_helpers` draw a plot or build an interactive demo. You never need to open that file.

**The pruning formula.** Cost-complexity pruning keeps the subtree with the lowest

$$
\text{total cost} = \text{training error} + \alpha \lvert T \rvert,
$$

where $\lvert T \rvert$ is the number of leaves: **training error + strength × complexity.** Bigger $\alpha$ means a simpler tree. Hold on to that shape.

In [ ]:
import numpy as np
import pandas as pd

<a id="section-1"></a>

## 1 · Same shape, new model: ridge and lasso

Least squares (Chapter 3) chooses the coefficients $\beta_0, \beta_1, \dots, \beta_p$ that make the RSS as small as possible. **Ridge regression** (ISL equation 6.5) adds a charge for big coefficients:

$$
\underbrace{\sum_{i=1}^{n} \Big( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \Big)^2}_{\text{RSS}} \;+\; \lambda \sum_{j=1}^{p} \beta_j^2
$$

In words: the RSS of the line, plus $\lambda$ times the sum of the squared coefficients. **The lasso** (ISL equation 6.7) is the same, except it charges the sum of the coefficients' absolute values:

$$
\sum_{i=1}^{n} \Big( y_i - \beta_0 - \sum_{j=1}^{p} \beta_j x_{ij} \Big)^2 \;+\; \lambda \sum_{j=1}^{p} \lvert \beta_j \rvert
$$

$\lambda \ge 0$ is the strength, chosen by us. Big coefficients let a linear model swing hard in response to small changes in a feature; charging for them keeps it calmer. This charge is called **shrinkage**, because it pulls the coefficients toward zero. (Ridge has a closed-form solution, like least squares, and scikit-learn uses it; the lasso is solved step by step.)

Here are the three penalties side by side. This table is the center of the week.

| Method | Training error | + | Strength × complexity | Strength: ISL / scikit-learn |
| --- | --- | --- | --- | --- |
| Pruning (eq. 8.4) | RSS of the tree | + | $\alpha \lvert T \rvert$: number of leaves | $\alpha$ / `ccp_alpha` |
| Ridge (eq. 6.5) | RSS of the line | + | $\lambda \sum_j \beta_j^2$: sum of squared coefficients | $\lambda$ / `alpha` |
| Lasso (eq. 6.7) | RSS of the line | + | $\lambda \sum_j \lvert \beta_j \rvert$: sum of absolute coefficients | $\lambda$ / `alpha` |

> **A note on symbols: watch the letters.** ISL and scikit-learn use different letters for these strengths, and scikit-learn reuses ISL's pruning letter for ridge and lasso.
>
> | What it controls | ISL writes | scikit-learn writes | Bigger value means |
> | --- | --- | --- | --- |
> | Pruning penalty | $\alpha$ (eq. 8.4) | `ccp_alpha` | **simpler** tree |
> | Ridge / lasso penalty | $\lambda$ (eqs. 6.5, 6.7) | `alpha` in `Ridge`, `Lasso` | **simpler** model (more shrinkage) |
> | Logistic regression penalty | — (ISL §6.2 is about regression) | `C` in `LogisticRegression` | **weaker** penalty |
>
> The last row is the one from Chapter 4: `LogisticRegression` adds a ridge-style penalty by default, with `C=1`. `C` is an *inverse* strength, so a smaller `C` means a *stronger* penalty, and `C=np.inf` turns it off, which is why the Chapter 4 notebooks used it.

**Ask:** At $\lambda = 0$, what model do ridge and lasso give? At a huge $\lambda$?

<details>
<summary>Answer</summary>

At $\lambda = 0$ there's no penalty, so both give ordinary least squares. At a huge $\lambda$, any nonzero coefficient costs more than it could save in RSS, so all the coefficients go to (nearly) zero, and the model predicts about the average response for everyone.

</details>

<a id="section-2"></a>

## 2 · Standardize first

The penalty charges every coefficient the same. But a coefficient's size depends on its feature's units. Measure a player's career at-bats in thousands instead of single at-bats, and its coefficient grows 1,000 times, so the penalty would charge it 1,000,000 times more (for ridge). **Standardizing** puts every feature in the same unit, standard deviations, so the penalty treats them fairly. You saw this in the gradient-descent notebook, where standardizing `TV` made the error bowl round, and in Chapter 4's optional lab.

The intercept $\beta_0$ isn't penalized (look at the sums: they start at $j = 1$). It only sets the average level of the predictions, not how hard the model swings.

Trees don't need standardizing. A split only asks "which side of $s$ is this point on?", and changing a feature's units moves $s$ along with the points.

**Code worth reading.** We use all of Hitters this time: 19 features to predict salary (in thousands of dollars), dropping the players with no salary, as in *Trees and Regularization*. Three columns are categories with two values each (`League`, `Division`, `NewLeague`).

- `make_column_transformer` applies different steps to different columns: here, `StandardScaler` to the numeric columns and `OneHotEncoder` to the categorical ones.
- `OneHotEncoder` turns a category into 0/1 columns. With `drop='if_binary'`, a two-valued column like `League` (A or N) becomes a single column, `League_N`, that is 1 for league N and 0 for league A.
- `make_pipeline` chains the preprocessing and the model, so `fit` runs every step in order, and cross-validation later repeats every step inside each fold.

In [ ]:
from ISLP import load_data
from sklearn.compose import make_column_transformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

hitters = load_data('Hitters').dropna(subset=['Salary'])
X = hitters.drop(columns='Salary')
y = hitters['Salary']                                # in thousands of dollars

categorical_columns = ['League', 'Division', 'NewLeague']
numeric_columns = [col for col in X.columns if col not in categorical_columns]

preprocess = make_column_transformer(
    (StandardScaler(), numeric_columns),
    (OneHotEncoder(drop='if_binary'), categorical_columns),
    verbose_feature_names_out=False)                 # keep the original column names
ridge = make_pipeline(preprocess, Ridge(alpha=10))
ridge.fit(X, y)

pd.Series(ridge[-1].coef_, index=preprocess.get_feature_names_out()).round(1)

`ridge[-1]` is the last step of the pipeline, the `Ridge` model, and `coef_` holds its coefficients. Each one is the change in predicted salary (in thousands) for a one-standard-deviation increase in that feature, with the others held fixed.

<a id="demo-f"></a>

## 3 · Shrinking a wiggly fit

### Demo F: $\lambda$ as a flexibility dial

Before the 19-feature version, a one-picture version. Here are 20 points from a smooth curve plus noise. The model is a line through 12 features made from $x$: $x, x^2, \dots, x^{12}$. With 12 features and 20 points, least squares can bend a lot. The right panel shows the 12 coefficients (on a scale that shows both huge and small values).

In [ ]:
from ch_6_and_8_demos_helpers import ridge_wiggle_demo

ridge_wiggle_demo(size_percent=80)

**Ask:** As I slide $\lambda$ up, what happens to the curve, and to the bars?

<details>
<summary>Answer</summary>

The bars shrink toward zero, and the curve calms down: from a wild curve that chases the points (and shoots off at the edges), to one close to the true curve, to a nearly flat line at very large $\lambda$. The error on new points falls, then rises: small $\lambda$ is the flexible end, large $\lambda$ the rigid end.

</details>

<a id="demo-g"></a>

## 4 · Watching coefficients shrink: ridge vs. lasso

### Demo G: Coefficient paths

Back to Hitters, with all 19 standardized features (ISL Figures 6.4 and 6.6). Each line is one coefficient, plotted against $\lambda$. Toggle between **Ridge** and **Lasso**, and watch the readout.

**Code worth reading.** `preprocess.fit_transform(X)` standardizes and encodes the features once, so the demo can fit many models quickly.

In [ ]:
X_std = preprocess.fit_transform(X)                  # standardized, one-hot encoded
feature_names = list(preprocess.get_feature_names_out())

In [ ]:
from ch_6_and_8_demos_helpers import coefficient_path_demo

coefficient_path_demo(X_std, y, feature_names, size_percent=80)

Ridge shrinks every coefficient toward zero, but none of them reach it: at every $\lambda$, all 19 features stay in the model. The lasso sets coefficients **exactly** to zero, one after another, as $\lambda$ grows. A feature with a zero coefficient is out of the model, so the lasso does **feature selection** as part of fitting. That's why it's called an *embedded* feature-selection method.

One more scikit-learn detail: `Lasso` divides the RSS by $2n$ before adding the penalty, and `Ridge` doesn't. So the same `alpha` number means a much stronger penalty in `Lasso` than in `Ridge`. Never copy a value from one to the other; choose each by cross-validation.

**Code worth reading.** Fit the lasso at one $\lambda$ and list the features it keeps.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=10).fit(X_std, y)
kept = pd.Series(lasso.coef_, index=feature_names)
print(f'Kept {np.sum(kept != 0)} of {len(kept)} features:')
kept[kept != 0].round(1)

**Ask:** You suspect only a handful of these 19 features matter. Which penalty would you try first, and why?

<details>
<summary>Answer</summary>

The lasso. It can drop the features that don't matter (set their coefficients to exactly 0), leaving a smaller model that's easier to read. Ridge would keep all 19. (If many features each matter a little, ridge often predicts better; cross-validation can settle it.)

</details>

<a id="demo-h"></a>

### Demo H (optional): Why the lasso hits zero

Picture just two coefficients, $\beta_1$ and $\beta_2$. The gray ovals are RSS contours, like the error bowls in the gradient-descent notebook; least squares sits at the bottom of the bowl. The penalty works like a budget: the coefficients must stay inside a region around zero. For ridge, that region is a circle ($\beta_1^2 + \beta_2^2$ stays small); for the lasso, a diamond ($\lvert \beta_1 \rvert + \lvert \beta_2 \rvert$ stays small). The best point is where the lowest contour first touches the region (ISL Figure 6.7).

In [ ]:
from ch_6_and_8_demos_helpers import plot_lasso_corner

plot_lasso_corner();

The diamond has corners on the axes, and the contours often touch a corner first. At a corner, one coefficient is exactly 0. The circle has no corners, so ridge's best point almost never lands exactly on an axis.

<a id="demo-i"></a>

## 5 · Choosing $\lambda$ by cross-validation

### Demo I: The cross-validation curve

**Code worth reading.** The same 10-fold `cross_val_score` loop as Lab 4, over a grid of $\lambda$ values. `np.logspace(-2, 5, 29)` makes 29 values from $10^{-2}$ to $10^{5}$, evenly spaced on a log scale. The whole pipeline goes into `cross_val_score`, so the standardizing is redone inside every fold, using only that fold's training players. `scoring='neg_mean_squared_error'` returns the MSE with a minus sign, so we flip it back, as in Lab 4.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=10, shuffle=True, random_state=0)
ridge_lambdas = np.logspace(-2, 5, 29)
lasso_lambdas = np.logspace(-2, 3, 21)

ridge_mse = []
for lam in ridge_lambdas:
    model = make_pipeline(preprocess, Ridge(alpha=lam))
    scores = cross_val_score(model, X, y, cv=folds, scoring='neg_mean_squared_error')
    ridge_mse.append(-scores.mean())

lasso_mse = []
for lam in lasso_lambdas:
    model = make_pipeline(preprocess, Lasso(alpha=lam, max_iter=100_000))   # more steps: fully solved
    scores = cross_val_score(model, X, y, cv=folds, scoring='neg_mean_squared_error')
    lasso_mse.append(-scores.mean())

**Code worth reading.** Two reference points. `LinearRegression` is least squares, no penalty ($\lambda = 0$). `DummyRegressor` ignores the features and predicts the training players' average salary for everyone, which is where a huge $\lambda$ ends up.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

ols_mse = -cross_val_score(make_pipeline(preprocess, LinearRegression()), X, y, cv=folds,
                           scoring='neg_mean_squared_error').mean()
mean_mse = -cross_val_score(DummyRegressor(), X, y, cv=folds,
                            scoring='neg_mean_squared_error').mean()

best_ridge, best_lasso = np.argmin(ridge_mse), np.argmin(lasso_mse)
pd.DataFrame({'10-fold CV MSE': [ols_mse, ridge_mse[best_ridge], lasso_mse[best_lasso], mean_mse]},
             index=['least squares (no penalty)',
                    f'ridge, best lambda = {ridge_lambdas[best_ridge]:.2g}',
                    f'lasso, best lambda = {lasso_lambdas[best_lasso]:.2g}',
                    'predict the average']).round(0)

In [ ]:
from ch_6_and_8_demos_helpers import plot_cv_lambda

plot_cv_lambda({'Ridge': (ridge_lambdas, ridge_mse), 'Lasso': (lasso_lambdas, lasso_mse)},
               ols_mse, mean_mse);

Both curves dip a little below least squares, then rise steeply toward "predict the average" as $\lambda$ grows. On all 263 players, the best penalty helps only a little, about 4%. Nineteen features and 263 players isn't much flexibility to tame. **Penalties help most when there are many features for the amount of data**, and that's exactly the situation of the neural networks coming up, which can have far more coefficients than training rows.

In bias–variance terms: on data like this, ridge with a large $\lambda$ has higher bias and lower variance; with a small $\lambda$, the reverse. Like $K$, $\alpha$, and `max_depth`, $\lambda$ is a flexibility setting.

**Ask:** Near the bottom, the ridge curve is flat across a range of $\lambda$. Does it matter which one you pick?

<details>
<summary>Answer</summary>

Not much. As with $K$ in Chapter 5, values near the flat bottom are about equally good, and which one wins can depend on the shuffle into folds. What cross-validation reliably does is steer you away from the bad end (here, very large $\lambda$).

</details>

**Code worth reading.** As always, cross-validation is for choosing. Refit the chosen model on all the players.

In [ ]:
final_lasso = make_pipeline(preprocess, Lasso(alpha=lasso_lambdas[best_lasso], max_iter=100_000))
final_lasso.fit(X, y)
n_zero = np.sum(final_lasso[-1].coef_ == 0)
print(f'Final lasso: lambda = {lasso_lambdas[best_lasso]:.2g}, {n_zero} of 19 features dropped')

<a id="section-6"></a>

## 6 · Ensembles: many trees

A single full tree is high variance on data like Chapter 5's. Pruning and limits fight variance by making the tree smaller. **Ensembles** fight it another way: grow many trees and combine them.

**Bagging** (ISL §8.2.1). Chapter 5 bootstrapped the data to see how much a number would change. Bagging bootstraps the *training data* to build many trees, then lets them vote:

1. Draw $B$ bootstrap samples of the training data (each the same size, drawn with replacement).
2. Grow a full tree on each.
3. Predict by averaging the $B$ trees' predictions (regression), or by their vote (classification).

ISL writes the average as $\hat f_{\text{bag}}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat f^{*b}(x)$: the average of the $B$ trees' predictions. Remember Demo E in Chapter 5, where the average of 50 wild $K = 1$ boundaries landed close to the Bayes boundary? With real data we couldn't draw 50 training sets. Bagging fakes it with bootstrap samples: each tree is wild, but their mistakes partly cancel.

**Random forests** (ISL §8.2.2). Bagged trees tend to make the same first split, on the strongest feature, so their mistakes are alike and don't cancel well. A random forest lets each split consider only a random subset of the features (`max_features` in scikit-learn), so the trees disagree more, and their average is steadier.

**Boosting** (ISL §8.2.3). Grow a small tree, see what it gets wrong, grow the next small tree to fix that, and add it in with a small step. Its three settings are the number of trees $B$ (`n_estimators`), the step size $\lambda$ (`learning_rate`), and the size of each tree (`max_depth`).

> **The third $\lambda$.** ISL calls boosting's step size $\lambda$ too (eq. 8.10, "shrinkage"). It's unrelated to the ridge/lasso $\lambda$: scikit-learn calls it `learning_rate`, and a bigger value means bigger steps.

scikit-learn's `HistGradientBoostingClassifier` is a fast version of boosting, and the one you'll likely use on Kaggle.

<a id="demo-j"></a>

### Demo J: Ensembles on the simulated data

**Code worth reading.** Five classifiers fit to Chapter 5's 200 training points and scored on its 10,000 test points. `BaggingClassifier(DecisionTreeClassifier(), n_estimators=300)` is bagging with 300 full trees. `RandomForestClassifier` is a random forest of full trees.

In [ ]:
from sklearn.ensemble import (BaggingClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.tree import DecisionTreeClassifier
from ch_6_and_8_demos_helpers import bayes_error_rate, simulate_data

sim = simulate_data()
X_sim, y_sim = sim.X, sim.y

models = {'One full tree': DecisionTreeClassifier(random_state=0),
          'Bagging (300 trees)': BaggingClassifier(DecisionTreeClassifier(), n_estimators=300,
                                                   random_state=0),
          'Random forest (300 trees)': RandomForestClassifier(n_estimators=300, random_state=0),
          'Gradient boosting': GradientBoostingClassifier(random_state=0),
          'Histogram gradient boosting': HistGradientBoostingClassifier(random_state=0)}
rows = []
for name, model in models.items():
    model.fit(X_sim, y_sim)
    rows.append({'model': name,
                 'training error': 1 - model.score(X_sim, y_sim),
                 'test error': 1 - model.score(sim.X_test, sim.y_test)})
print(f'Bayes error rate: {bayes_error_rate(sim):.1%}')
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
from ch_6_and_8_demos_helpers import plot_ensemble_boundaries

shown = ['One full tree', 'Bagging (300 trees)', 'Random forest (300 trees)',
         'Histogram gradient boosting']
plot_ensemble_boundaries(sim, {name: models[name] for name in shown});

Every ensemble beats the single full tree on new data, even though most of them still get every training point right. Their boundaries are still made of boxes, but the boxes average out into a shape closer to the Bayes boundary.

With only two features, a random forest's "random subset" is one feature per split, so it barely differs from bagging here. The idea matters more with many features, as you'll see in the lab.

Slide $B$ to watch a bag of trees settle down as trees are added.

In [ ]:
from ch_6_and_8_demos_helpers import bagging_demo

bagging_demo(sim, size_percent=80)

**Ask:** Adding more trees to a random forest never makes it overfit more. Adding more trees to boosting can. Why the difference?

<details>
<summary>Answer</summary>

Each forest tree is fit independently, on its own bootstrap sample, so adding trees just makes the average steadier. Each boosting tree is fit to what the trees before it still get wrong on the training data. Keep adding trees and boosting keeps chasing the remaining training errors, including the noise. So for boosting, the number of trees is a flexibility setting to choose by cross-validation; for a forest, more trees only costs time.

</details>

<a id="section-7"></a>

## 7 · Summary, and next week

- Pruning, ridge, and lasso share one shape: **training error + strength × complexity**. Complexity is the number of leaves for a tree, and the size of the coefficients for a line.
- Ridge shrinks every coefficient toward zero; the lasso shrinks them and sets some exactly to zero, so it also selects features.
- Standardize the features before ridge or lasso, so the penalty treats them fairly. Trees don't need it.
- Choose the strength by cross-validation, then refit on all the data. Watch the letters: ISL's $\lambda$ is scikit-learn's `alpha`, and `C` runs the other way.
- Ensembles cut variance by averaging many trees (bagging, random forests), or build a strong model out of many small ones (boosting, where the number of trees must be chosen).
- **Next week:** a neural network has thousands of coefficients. The most common way to keep them in check, *weight decay*, is exactly ridge's penalty added to the network's training error.

# Practice Questions

These questions cover ridge and lasso, standardizing, choosing $\lambda$, the notation, and ensembles. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

For reference:

- Ridge: RSS $+ \lambda \sum_j \beta_j^2$. Lasso: RSS $+ \lambda \sum_j \lvert \beta_j \rvert$. Pruning: training error $+ \alpha \lvert T \rvert$.

### Question 1: Same shape

Match each complexity charge to its method: pruning, ridge, or lasso. (See [Section 1](#section-1).)

(a) $\lambda \sum_j \lvert \beta_j \rvert$ &emsp; (b) $\alpha \lvert T \rvert$ &emsp; (c) $\lambda \sum_j \beta_j^2$

<details>
<summary>Answer</summary>

(a) **Lasso** (sum of absolute coefficients). (b) **Pruning** (number of leaves). (c) **Ridge** (sum of squared coefficients).

</details>

### Question 2: The ends of the dial

(a) What model does ridge regression give at $\lambda = 0$?

(b) What does it predict, approximately, at a huge $\lambda$? (See [Section 1](#section-1).)

<details>
<summary>Answer</summary>

(a) **Ordinary least squares:** there's no penalty.

(b) **About the average response for everyone:** every coefficient is shrunk to nearly zero, leaving only the intercept, which isn't penalized.

</details>

### Question 3: Ridge or lasso?

(a) Which method can set coefficients exactly to zero?

(b) After fitting at some $\lambda$, a model has 7 of its 19 coefficients exactly equal to 0. Was it ridge or the lasso?

(c) What is this "free" feature selection called, as a family of feature-selection methods? (See [Demo G](#demo-g).)

<details>
<summary>Answer</summary>

(a) **The lasso.** (b) **The lasso:** ridge shrinks coefficients toward zero but doesn't set them exactly to zero. (c) **An embedded method:** the selection happens as part of fitting.

</details>

### Question 4: Standardize first

(a) A feature is measured in dollars. You change it to thousands of dollars. What happens to its least squares coefficient?

(b) Why does that matter for ridge and lasso, and what do we do about it?

(c) Does a decision tree need standardized features? (See [Section 2](#section-2).)

<details>
<summary>Hint</summary>

A coefficient is the change in the prediction per one unit of the feature. How big is one unit now?

</details>

<details>
<summary>Answer</summary>

(a) **It gets 1,000 times bigger**, because one unit of the feature is now 1,000 times bigger.

(b) The penalty charges every coefficient the same, so a feature's units would decide how hard it gets penalized. **Standardize** every feature first, so each is measured in standard deviations.

(c) **No.** A split only asks which side of $s$ a point is on, and changing units moves $s$ along with the points.

</details>

### Question 5: Translate the letters

For each, give the scikit-learn name. (See the note on symbols in [Section 1](#section-1).)

(a) ISL's pruning strength $\alpha$.

(b) ISL's ridge strength $\lambda$.

(c) ISL's boosting step size $\lambda$.

(d) True or false: in `LogisticRegression`, making `C` smaller makes the penalty weaker.

<details>
<summary>Answer</summary>

(a) **`ccp_alpha`.** (b) **`alpha`** (in `Ridge`). (c) **`learning_rate`.**

(d) **False.** `C` is an inverse strength: a smaller `C` means a *stronger* penalty.

</details>

### Question 6: Choose $\lambda$

10-fold cross-validation for the lasso gives:

| $\lambda$ | 0.1 | 1 | 3 | 10 | 100 |
| --- | --- | --- | --- | --- | --- |
| CV MSE | 116,000 | 113,500 | 111,700 | 115,400 | 136,400 |

(a) Which $\lambda$ would you choose?

(b) What do you do next, before predicting new players' salaries? (See [Demo I](#demo-i).)

<details>
<summary>Answer</summary>

(a) **$\lambda = 3$**, the lowest CV MSE.

(b) **Refit the lasso with $\lambda = 3$ on all the training data**, and use that model. Cross-validation was only for choosing.

</details>

### Question 7: $\lambda$ and bias–variance

On data like Hitters, which has higher bias, ridge with $\lambda = 1{,}000$ or ridge with $\lambda = 0.01$? Which has higher variance? (See [Demo I](#demo-i).)

<details>
<summary>Answer</summary>

**$\lambda = 1{,}000$ has higher bias; $\lambda = 0.01$ has higher variance.** A large $\lambda$ shrinks the coefficients hard, so the method is rigid; a tiny $\lambda$ is nearly least squares, the flexible end.

</details>

### Question 8: Bagging

(a) Describe bagging in three steps.

(b) Why does averaging many trees give lower test error than one full tree? (See [Section 6](#section-6).)

<details>
<summary>Hint</summary>

Think back to Chapter 5's Demo E and the average of many $K = 1$ boundaries.

</details>

<details>
<summary>Answer</summary>

(a) **Draw $B$ bootstrap samples of the training data; grow a full tree on each; average their predictions (or take a vote).**

(b) Each full tree is high variance: its mistakes depend on its particular sample. The trees' mistakes point in different directions and partly cancel when averaged, so the bag's predictions change much less from sample to sample. **Averaging reduces variance.**

</details>

### Question 9: Forest or boosting?

(a) What does a random forest do that bagging doesn't?

(b) Which can overfit as you add more trees: a random forest or boosting? Why? (See [Demo J](#demo-j).)

<details>
<summary>Answer</summary>

(a) **Each split considers only a random subset of the features** (`max_features`), so the trees disagree more and their average is steadier.

(b) **Boosting.** Each new boosting tree is fit to what the earlier trees still get wrong on the training data, so with enough trees it chases noise. A forest's trees are fit independently, so more trees only steady the average.

</details>

### Question 10: Next week

A neural network is trained by minimizing its training error plus $\lambda$ times the sum of its squared weights. Which method from this notebook uses the same penalty? (See [Section 7](#section-7).)

<details>
<summary>Answer</summary>

**Ridge regression.** In neural networks, this penalty is called *weight decay*.

</details>